# Exercise 11 — Frank-Wolfe Algorithm for Sparse Regression

---

## Introduction

Many machine learning tasks require *sparse* models — weight vectors where most coefficients equal zero.
A natural approach is to constrain the **ℓ₁-norm** of the weight vector:

$$\min_{\mathbf{w} \in \mathcal{D}}\; f(\mathbf{w}), \qquad \mathcal{D} = \bigl\{\mathbf{w} \in \mathbb{R}^d \mid \|\mathbf{w}\|_1 \leq t\bigr\},$$

where $f(\mathbf{w}) = \dfrac{1}{2n}\|X\mathbf{w} - \mathbf{y}\|_2^2$ is the mean squared error (MSE) of a linear predictor.
This is the **constrained LASSO** (basis-pursuit) formulation.

### Two algorithms compared

| | **Projected Gradient Descent (PGD)** | **Frank-Wolfe (FW)** |
|---|---|---|
| Key operation | Gradient step + **Projection** onto $\mathcal{D}$ | Gradient step + **LMO** over $\mathcal{D}$ |
| Per-iteration cost | $O(nd + d\log d)$ | $O(nd)$ |
| Stays feasible by | Explicit projection | Convex combinations |
| Sparse iterates | Not guaranteed | ✓ Automatically sparse |

### Frank-Wolfe update rule

Starting from $\mathbf{w}_0 \in \mathcal{D}$, at each step $k = 0, 1, 2, \ldots$:

$$\underbrace{s_k \;=\; \arg\min_{s \in \mathcal{D}}\; \nabla f(\mathbf{w}_k)^\top s}_{\text{Linear Minimization Oracle (LMO)}}$$

$$\mathbf{w}_{k+1} = (1 - \gamma_k)\,\mathbf{w}_k + \gamma_k\, s_k, \qquad \gamma_k = \frac{2}{k+2}$$

Because the update is a **convex combination**, $\mathbf{w}_{k+1} \in \mathcal{D}$ automatically — **no projection needed!**

### Convergence rate

For a convex, $L$-smooth objective over a bounded convex domain $\mathcal{D}$
(Theorem 57 in the lecture notes):

$$f(\mathbf{w}_k) - f^\star \;\leq\; \frac{2M}{k+2},$$

where the **curvature constant** satisfies $M \leq \operatorname{diam}^2(\mathcal{D}) \cdot L$.
For the ℓ₁ ball, $\operatorname{diam}(\mathcal{D}) = 2t$, so $M \leq 4t^2 L$.

### Learning objectives
**(§2.1)** Implement the **LMO** for the ℓ₁ ball.
**(§2.2)** Compute the **duality gap** as a parameter-free optimality certificate.
**(§3)**   Implement **Frank-Wolfe** (conditional gradient) from scratch.
**(§4)**   Implement **Projected Gradient Descent** and contrast it with FW.
**(§5)**   Visualise the **2-D geometry**: trajectories on the ℓ₁ diamond.
**(§6)**   Plot **learning curves** and verify the $O(1/k)$ rate vs the theory.
**(§7)**   Use the **duality gap** as a practical stopping criterion.
**(§8)**   Explore the **sparsity–accuracy trade-off** by varying the budget $t$.

## 0 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

rng = np.random.default_rng(42)

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

---
## 1 — Problem Setup

We minimise the **mean squared error** subject to an **ℓ₁-ball constraint**:

$$f(\mathbf{w}) = \frac{1}{2n}\|X\mathbf{w} - \mathbf{y}\|_2^2, \qquad
  \nabla f(\mathbf{w}) = \frac{1}{n}\,X^\top(X\mathbf{w} - \mathbf{y}).$$

The **smoothness constant** of $f$ is $L = \lambda_{\max}(X^\top X)/n$
(largest eigenvalue of the Gram matrix divided by $n$).

The cell below provides:
- `f(w, X, y)` — objective value
- `grad_f(w, X, y)` — gradient $\nabla f(\mathbf{w})$
- `smoothness_constant(X)` — computes $L$
- `make_data(n, d, ...)` — generates a synthetic sparse regression dataset

In [ ]:
def f(w, X, y):
    """Least-squares objective: f(w) = (1/2n) ||Xw - y||^2"""
    r = X @ w - y
    return 0.5 * np.dot(r, r) / len(y)


def grad_f(w, X, y):
    """Gradient: nabla f(w) = (1/n) X^T (Xw - y)"""
    return X.T @ (X @ w - y) / len(y)


def smoothness_constant(X):
    """Smoothness constant L = lambda_max(X^T X) / n  (= ||X||_2^2 / n)."""
    return (np.linalg.norm(X, ord=2) ** 2) / X.shape[0]


def make_data(n, d, n_nonzero=5, noise=0.1, seed=42):
    """
    Generate synthetic sparse regression data.

    Returns
    -------
    X      : (n, d) column-normalised feature matrix
    y      : (n,) target vector
    w_true : (d,) true sparse weight vector
    """
    rng_local = np.random.default_rng(seed)
    X = rng_local.standard_normal((n, d))
    X /= np.linalg.norm(X, axis=0, keepdims=True)   # column-normalise
    w_true = np.zeros(d)
    idx = rng_local.choice(d, min(n_nonzero, d), replace=False)
    w_true[idx] = rng_local.standard_normal(len(idx))
    y = X @ w_true + noise * rng_local.standard_normal(n)
    return X, y, w_true


# ── Sanity check ─────────────────────────────────────────────────────────────
Xs, ys, ws = make_data(n=50, d=10)
w0 = np.zeros(10)
print(f"f(0)          = {f(w0, Xs, ys):.4f}")
print(f"||nabla f(0)|| = {np.linalg.norm(grad_f(w0, Xs, ys)):.4f}")
print(f"L              = {smoothness_constant(Xs):.4f}")
print(f"||w_true||_1   = {np.sum(np.abs(ws)):.4f}  (l1 norm of the true weights)")

---
## 2 — Linear Minimization Oracle and Duality Gap

### 2.1  LMO for the ℓ₁ ball

For $\mathcal{D} = \{\mathbf{w} : \|\mathbf{w}\|_1 \leq t\}$, the LMO solves:

$$s^\star = \arg\min_{\|s\|_1 \leq t}\; \nabla f(\mathbf{w})^\top s.$$

The **extreme points** (vertices) of the ℓ₁ ball are $\pm t\,\mathbf{e}_i$ for $i = 1, \ldots, d$.
Since we minimise a *linear* function, the minimum is always attained at a vertex:

$$\boxed{i^\star = \arg\max_i\;|\nabla_i f(\mathbf{w})|,
\qquad s^\star = -\,t\;\operatorname{sign}\!\left(\nabla_{i^\star} f(\mathbf{w})\right)\mathbf{e}_{i^\star}}$$

> **Key insight:** The LMO for the ℓ₁ ball is a *coordinate descent* step — it picks the single most
> important feature direction. This is why Frank-Wolfe automatically produces **sparse iterates**.

### 2.2  Duality gap

The **duality gap** at $\mathbf{w}_k$ is:

$$g(\mathbf{w}_k) = \nabla f(\mathbf{w}_k)^\top (\mathbf{w}_k - s_k)
= \nabla f(\mathbf{w}_k)^\top \mathbf{w}_k - \min_{s \in \mathcal{D}} \nabla f(\mathbf{w}_k)^\top s.$$

It satisfies $g(\mathbf{w}_k) \geq f(\mathbf{w}_k) - f^\star \geq 0$ (Proposition 56 in the lecture notes),
making it a **parameter-free stopping criterion** that does not require knowing $f^\star$.

In [ ]:
def lmo_l1(grad, t):
    """
    Linear Minimization Oracle for the l1 ball  {s : ||s||_1 <= t}.

    Returns  s* = argmin_{||s||_1 <= t}  grad^T s

    Parameters
    ----------
    grad : (d,) gradient vector nabla f(w)
    t    : positive constraint radius

    Returns
    -------
    s : (d,) extreme point of the l1 ball  (a vertex  +-t * e_{i*})
    """
    # ── TODO 2.1 ─────────────────────────────────────────────────────────────
    # The extreme points of {s : ||s||_1 <= t} are  +-t * e_i  for i=1,...,d.
    # The minimiser of  grad^T s  over these vertices is the vertex that most
    # reduces the objective, i.e.:
    #   s = -t * sign(grad[i*]) * e_{i*},  where  i* = argmax_i |grad[i]|.
    #
    # i_star = ???          # index of the largest-magnitude gradient component
    # s      = np.zeros_like(grad)
    # s[i_star] = ???       # -t * sign(grad[i_star])   (use np.sign)
    # return s
    # ─────────────────────────────────────────────────────────────────────────
    pass


# ── Sanity check ─────────────────────────────────────────────────────────────
g_test = np.array([0.3, -0.8, 0.5, -0.1])
t_test = 2.0
s_test = lmo_l1(g_test, t_test)
print(f"grad          = {g_test}")
print(f"LMO solution  = {s_test}")
print(f"||s||_1       = {np.sum(np.abs(s_test)) if s_test is not None else 'n/a'}  (should be {t_test})")
# Expected: i*=1 (|grad[1]|=0.8 largest), s = [0, 2, 0, 0],  grad^T s = -1.6
print(f"grad^T s      = {g_test @ s_test if s_test is not None else 'n/a'}  (should be {-t_test * np.max(np.abs(g_test)):.2f})")

In [ ]:
def duality_gap(w, grad, t):
    """
    Duality gap  g(w) = nabla f(w)^T (w - s*)  where  s* = LMO solution.

    This provides an upper bound on f(w) - f* and serves as a stopping criterion.

    Parameters
    ----------
    w    : (d,) current iterate
    grad : (d,) gradient nabla f(w)
    t    : constraint radius

    Returns
    -------
    gap : scalar >= 0
    """
    # ── TODO 2.2 ─────────────────────────────────────────────────────────────
    # s   = ???        # LMO solution  (use lmo_l1)
    # return ???       # grad^T (w - s)   — should always be >= 0
    # ─────────────────────────────────────────────────────────────────────────
    pass


# ── Sanity check ─────────────────────────────────────────────────────────────
# At w = 0:  gap = grad^T (0 - s*) = -grad^T s* = t * ||grad||_inf
X_s, y_s, _ = make_data(n=20, d=5, seed=1)
t_s  = 1.0
w_s  = np.zeros(5)
g_s  = grad_f(w_s, X_s, y_s)
gap  = duality_gap(w_s, g_s, t_s)
expected = t_s * np.max(np.abs(g_s))
print(f"Duality gap at w=0    : {gap}")
print(f"t * ||grad||_inf      : {expected:.6f}  (should match)")

---
## 3 — Frank-Wolfe Algorithm

The Frank-Wolfe algorithm (also called the *Conditional Gradient* method) iterates:

| Step | Operation |
|------|----------|
| 1 | Compute gradient $g_k = \nabla f(\mathbf{w}_k)$ |
| 2 | **LMO**: $s_k = \arg\min_{\|s\|_1 \leq t}\; g_k^\top s$ |
| 3 | **Step size**: $\gamma_k = \dfrac{2}{k+2}$ |
| 4 | **Update**: $\mathbf{w}_{k+1} = (1-\gamma_k)\,\mathbf{w}_k + \gamma_k\,s_k$ |

Starting point: $\mathbf{w}_0 = \mathbf{0}$ (feasible since $\|\mathbf{0}\|_1 = 0 \leq t$).

> **✏️ Exercise 3** — Implement the `frank_wolfe` function below.

In [ ]:
def frank_wolfe(X, y, t, n_iter=500, return_path=False):
    """
    Frank-Wolfe (Conditional Gradient) for constrained least-squares regression.

    Solves:  min_{||w||_1 <= t}  f(w) = (1/2n) ||Xw - y||^2

    Parameters
    ----------
    X           : (n, d) feature matrix
    y           : (n,) target vector
    t           : l1 constraint radius
    n_iter      : number of iterations
    return_path : if True, also return the array of iterates

    Returns
    -------
    losses : list of f(w_k) for k = 0 .. n_iter
    gaps   : list of duality gap g(w_k) for k = 0 .. n_iter
    path   : (n_iter+1, d) array of iterates  [only if return_path=True]
    """
    d = X.shape[1]
    w = np.zeros(d)   # initial iterate (feasible: ||0||_1 = 0 <= t)

    g      = grad_f(w, X, y)
    losses = [f(w, X, y)]
    gaps   = [duality_gap(w, g, t)]
    path   = [w.copy()] if return_path else None

    for k in range(n_iter):
        g = grad_f(w, X, y)

        # ── TODO 3 ──────────────────────────────────────────────────────────
        # Step 1 — step size  (default schedule: gamma_k = 2 / (k + 2))
        # gamma = ???
        #
        # Step 2 — LMO: find the extreme point that minimises g^T s
        # s = ???   (use lmo_l1)
        #
        # Step 3 — Frank-Wolfe update (convex combination of w and s)
        # w = ???
        # ────────────────────────────────────────────────────────────────────

        losses.append(f(w, X, y))
        gaps.append(duality_gap(w, grad_f(w, X, y), t))
        if return_path:
            path.append(w.copy())

    if return_path:
        return losses, gaps, np.array(path)
    return losses, gaps

---
## 4 — Projected Gradient Descent (for comparison)

Projected Gradient Descent (PGD) performs the update:

$$\mathbf{w}_{k+1} = \Pi_{\mathcal{D}}\!\left(\mathbf{w}_k - \eta\,\nabla f(\mathbf{w}_k)\right),$$

where $\Pi_{\mathcal{D}}$ is the Euclidean projection onto $\mathcal{D} = \{\mathbf{w} : \|\mathbf{w}\|_1 \leq t\}$
and the step size $\eta = 1/L$ is optimal for $L$-smooth objectives.

The projection onto the ℓ₁ ball requires $O(d \log d)$ (sorting-based algorithm, provided below).

> **✏️ Exercise 4** — Implement the `pgd` function using the provided `proj_l1`.

In [ ]:
def proj_l1(v, t):
    """
    Project v onto the l1 ball  {w : ||w||_1 <= t}.

    Uses the O(d log d) sorting-based algorithm of Duchi et al. (2008).
    No TODOs — provided for you.
    """
    if np.sum(np.abs(v)) <= t:
        return v.copy()
    u    = np.sort(np.abs(v))[::-1]          # sort |v| in descending order
    cssv = np.cumsum(u)
    rho  = np.where(u > (cssv - t) / np.arange(1, len(u) + 1))[0][-1]
    theta = (cssv[rho] - t) / (rho + 1.0)
    return np.sign(v) * np.maximum(np.abs(v) - theta, 0.0)


# ── Quick test ────────────────────────────────────────────────────────────────
v_test = np.array([3.0, -1.0, 0.5, -0.2])
p_test = proj_l1(v_test, 2.0)
print(f"proj = {p_test}")
print(f"||proj||_1 = {np.sum(np.abs(p_test)):.4f}  (should equal 2.0 — input is outside the ball)")

In [ ]:
def pgd(X, y, t, eta=None, n_iter=500, return_path=False):
    """
    Projected Gradient Descent for constrained least-squares regression.

    Solves:  min_{||w||_1 <= t}  f(w) = (1/2n) ||Xw - y||^2

    Parameters
    ----------
    X           : (n, d) feature matrix
    y           : (n,) target vector
    t           : l1 constraint radius
    eta         : step size; defaults to 1/L (optimal for L-smooth objectives)
    n_iter      : number of iterations
    return_path : if True, also return the array of iterates

    Returns
    -------
    losses : list of f(w_k) for k = 0 .. n_iter
    path   : (n_iter+1, d) array of iterates  [only if return_path=True]
    """
    n, d = X.shape
    if eta is None:
        L   = smoothness_constant(X)
        eta = 1.0 / L

    w      = np.zeros(d)
    losses = [f(w, X, y)]
    path   = [w.copy()] if return_path else None

    for _ in range(n_iter):
        # ── TODO 4 ──────────────────────────────────────────────────────────
        # g = ???        # compute gradient  nabla f(w)
        # w = ???        # gradient step followed by projection onto {||w||_1 <= t}
        #                #   (use proj_l1)
        # ────────────────────────────────────────────────────────────────────

        losses.append(f(w, X, y))
        if return_path:
            path.append(w.copy())

    if return_path:
        return losses, np.array(path)
    return losses

---
## 5 — Geometric Visualization in 2-D

To develop intuition for the **geometry** of both algorithms, we visualise their iterate
trajectories in 2-D.

- The **ℓ₁ ball** $\{\mathbf{w} : |w_1| + |w_2| \leq t\}$ is a **diamond** in 2-D.
- The **level sets** of $f(\mathbf{w})$ are **ellipses** centred at the unconstrained optimum.
- **Frank-Wolfe** moves toward a single **vertex** of the diamond at each step.
- **PGD** takes a gradient step and then **projects** back onto the diamond.

Key geometric observations:
- FW steps are *convex combinations* → they stay inside the diamond.
  The direction from $\mathbf{w}_k$ to $s_k$ is always toward a vertex.
- PGD steps can leave the feasible set and require explicit projection.
- Because FW always moves toward a **vertex** ($\pm t\,\mathbf{e}_i$), iterates tend to be sparse.

> **✏️ Exercise 5** — Complete the trajectory-plotting code below.
> After running the cell, describe the qualitative difference between the FW and PGD trajectories.

In [ ]:
# ── 2-D problem setup ─────────────────────────────────────────────────────────
n2, d2 = 40, 2
X2, y2, w_true2 = make_data(n=n2, d=d2, n_nonzero=2, noise=0.15, seed=7)
t2      = 0.8    # constraint radius; ||w_true||_1 > t2 → constrained opt. is on boundary
n_iter2 = 50

# Run FW and PGD (requires frank_wolfe and pgd to be implemented above)
losses_fw2,  gaps_fw2,  path_fw2  = frank_wolfe(X2, y2, t2, n_iter=n_iter2, return_path=True)
losses_pgd2,            path_pgd2 = pgd(X2, y2, t2, n_iter=n_iter2, return_path=True)

# Approximate constrained optimum w* via long FW run
_, _, path_long2 = frank_wolfe(X2, y2, t2, n_iter=3000, return_path=True)
w_star2 = path_long2[-1]
print(f"||w_star||_1 = {np.sum(np.abs(w_star2)):.4f}  (should be <= {t2})")
print(f"w_true       = {w_true2}")
print(f"w_star       = {w_star2}")

# ── Constraint set and objective contours ─────────────────────────────────────
xlim, ylim = 1.1, 1.1
w1g = np.linspace(-xlim, xlim, 300)
w2g = np.linspace(-ylim, ylim, 300)
W1, W2 = np.meshgrid(w1g, w2g)
Z = np.array([[f(np.array([a, b]), X2, y2) for a in w1g] for b in w2g])

diamond_pts = np.array([[t2, 0], [0, t2], [-t2, 0], [0, -t2], [t2, 0]])

fig, ax = plt.subplots(figsize=(8, 7))

# Filled feasible region (l1 ball)
ax.fill(diamond_pts[:, 0], diamond_pts[:, 1],
        alpha=0.15, color='steelblue',
        label=r'$\mathcal{D}=\{\|\mathbf{w}\|_1 \leq t\}$')
ax.plot(diamond_pts[:, 0], diamond_pts[:, 1], 'b-', lw=1.5, alpha=0.45)

# Contour lines of the objective
ax.contour(W1, W2, Z, levels=20, cmap='Oranges', alpha=0.6)

# Constrained optimum
ax.plot(*w_star2, 'k*', ms=15, zorder=7, label=r'$\mathbf{w}^\star$')

# ── TODO 5 ──────────────────────────────────────────────────────────────────
# Plot the FW trajectory (path_fw2) and PGD trajectory (path_pgd2).
# Each path is an (n_iter2+1, 2) array of 2-D iterates.
#
# (a) FW trajectory:
#     ax.plot(path_fw2[:, 0], path_fw2[:, 1], 'o-', color='tomato',
#             ms=4, lw=1.5, zorder=4, label='Frank-Wolfe')
#
# (b) PGD trajectory:
#     ax.plot(path_pgd2[:, 0], path_pgd2[:, 1], 's-', color='seagreen',
#             ms=4, lw=1.5, zorder=4, label='PGD')
#
# (c) [Bonus] For the first K_show = 8 FW steps, draw a dashed arrow from
#     path_fw2[k] to the LMO solution s_k to show the FW direction:
#
#     K_show = 8
#     for k in range(K_show):
#         gk = grad_f(path_fw2[k], X2, y2)
#         sk = lmo_l1(gk, t2)
#         ax.annotate('', xy=sk, xytext=path_fw2[k],
#                     arrowprops=dict(arrowstyle='->', color='tomato',
#                                     lw=0.9, linestyle='dashed', alpha=0.55))
# ─────────────────────────────────────────────────────────────────────────────

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title(r'Trajectories on the $\ell_1$ Ball  ($t = $' + f'{t2})')
ax.legend(loc='upper right')
ax.set_xlim(-xlim, xlim)
ax.set_ylim(-ylim, ylim)
plt.tight_layout()

---
## 6 — Learning Curves on a Higher-Dimensional Problem

We now run both methods on a higher-dimensional problem ($n=100$, $d=50$) and compare:
1. The **suboptimality** $f(\mathbf{w}_k) - f^\star$ on a log scale.
2. The **theoretical bound** $\dfrac{2M}{k+2}$ with $M \leq 4t^2 L$.

This lets us verify the $O(1/k)$ convergence rate of Frank-Wolfe.

> **Note:** PGD with step size $\eta = 1/L$ achieves an $O(1/k)$ rate for convex $L$-smooth functions,
> but with a potentially smaller constant (as there is no curvature constant $M$). Compare empirically!

> **✏️ Exercise 6** — Run both methods and produce the convergence plot.

In [ ]:
# ── Problem setup ─────────────────────────────────────────────────────────────
n, d    = 100, 50
X, y, w_true = make_data(n=n, d=d, n_nonzero=5, noise=0.1, seed=42)
t       = 1.5
n_iter  = 500

# Smoothness constant and curvature bound
L_val = smoothness_constant(X)
M_val = (2 * t) ** 2 * L_val    # M <= diam^2(D) * L,  diam(l1 ball) = 2t
print(f"L = {L_val:.4f},   diam(D) = {2*t:.1f},   M <= {M_val:.4f}")

# Approximate f_star via long runs
_l_fw_long, _  = frank_wolfe(X, y, t, n_iter=5000)
_l_pgd_long    = pgd(X, y, t, n_iter=5000)
f_star = min(min(_l_fw_long), min(_l_pgd_long))
print(f"f_star ≈ {f_star:.6f}")

# ── TODO 6 ──────────────────────────────────────────────────────────────────
# 1. Run frank_wolfe and pgd for n_iter iterations:
#    losses_fw, gaps_fw = frank_wolfe(X, y, t, n_iter=n_iter)
#    losses_pgd         = pgd(X, y, t, n_iter=n_iter)
#
# 2. Compute the theoretical upper bound  2*M / (k+2)  for k = 0 .. n_iter:
#    k_arr  = np.arange(n_iter + 1)
#    theory = ???
#
# 3. Plot suboptimality on a semilogy axes:
#    fig, ax = plt.subplots(figsize=(9, 4))
#    ax.semilogy(np.array(losses_fw)  - f_star, color='tomato',   lw=1.8, label='Frank-Wolfe')
#    ax.semilogy(np.array(losses_pgd) - f_star, color='seagreen', lw=1.8, label='PGD')
#    ax.semilogy(theory, color='gray', ls='--', lw=1.5,
#                label=r'$2M\,/\,(k+2)$  (FW theory)')
#    ax.set_xlabel('Iteration $k$')
#    ax.set_ylabel(r'$f(\mathbf{w}_k) - f^\star$')
#    ax.set_title(r'Learning Curves: Frank-Wolfe vs PGD  ($n=100,\;d=50,\;t=1.5$)')
#    ax.legend()
#    plt.tight_layout()
# ─────────────────────────────────────────────────────────────────────────────

---
## 7 — Duality Gap as a Convergence Certificate

Recall that the duality gap $g(\mathbf{w}_k)$ satisfies:

$$g(\mathbf{w}_k) \;=\; \nabla f(\mathbf{w}_k)^\top (\mathbf{w}_k - s_k)
\;\geq\; f(\mathbf{w}_k) - f^\star \;\geq\; 0.$$

**Advantages:**
- It is computed *for free* inside the Frank-Wolfe loop (no extra gradient evaluation).
- It does not require knowledge of $f^\star$ (unlike the suboptimality plot).
- It can be used as a **practical stopping criterion**: stop when $g(\mathbf{w}_k) \leq \varepsilon$.

> **✏️ Exercise 7** — Plot the duality gap alongside the true suboptimality to verify
> the inequality $g(\mathbf{w}_k) \geq f(\mathbf{w}_k) - f^\star$.

In [ ]:
# ── TODO 7 ──────────────────────────────────────────────────────────────────
# Using the variables losses_fw, gaps_fw, and f_star from Exercise 6,
# plot the duality gap and the true suboptimality on the same axes.
#
# fig, ax = plt.subplots(figsize=(9, 4))
# ax.semilogy(np.array(losses_fw) - f_star,
#             color='tomato', lw=1.8, label=r'$f(\mathbf{w}_k) - f^\star$  (true)')
# ax.semilogy(gaps_fw,
#             color='tomato', ls='--', lw=1.5, label=r'Duality gap  $g(\mathbf{w}_k)$')
# ax.set_xlabel('Iteration $k$')
# ax.set_ylabel('Value')
# ax.set_title('Duality Gap vs True Suboptimality  (Frank-Wolfe)')
# ax.legend()
# plt.tight_layout()
#
# Does the duality gap always lie above the true suboptimality?
# How tight is the bound?
# ─────────────────────────────────────────────────────────────────────────────

---
## 8 — Effect of the Constraint Budget $t$ (Bonus)

Varying the ℓ₁ constraint radius $t$ controls the **sparsity–accuracy trade-off**:
- **Small $t$**: tight constraint → very sparse solution → higher training error.
- **Large $t$**: loose constraint → denser solution → lower training error.

When $t \geq \|\mathbf{w}_{\text{OLS}}\|_1$, the constraint is inactive and FW recovers
the ordinary least-squares solution.

> **✏️ Exercise 8 (Bonus)** — Sweep $t$ and visualise the sparsity–accuracy trade-off.

In [ ]:
# ── TODO 8 (Bonus) ──────────────────────────────────────────────────────────
# For each t_val in t_values:
#   1. Run frank_wolfe(X, y, t_val, n_iter=500) to obtain the final iterate w_final.
#   2. Record:
#      - training_losses : f(w_final, X, y)
#      - n_nonzeros      : number of entries with |w_final[i]| > 1e-3
#
# Then make a 2-panel figure:
#   Panel 1 (semilogy): training loss vs t
#   Panel 2           : number of non-zero coefficients vs t
#
# t_values        = np.linspace(0.1, 5.0, 25)
# training_losses = []
# n_nonzeros      = []
#
# for t_val in t_values:
#     losses_t, _ = frank_wolfe(X, y, t_val, n_iter=500)
#     w_final = ???
#     training_losses.append(???)
#     n_nonzeros.append(???)
#
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
# ax1.semilogy(t_values, training_losses, 'o-', color='steelblue')
# ax1.set_xlabel('Constraint budget $t$')
# ax1.set_ylabel(r'Training loss $f(\mathbf{w}_{500})$')
# ax1.set_title('Training loss vs constraint budget')
# ax2.plot(t_values, n_nonzeros, 'o-', color='tomato')
# ax2.set_xlabel('Constraint budget $t$')
# ax2.set_ylabel('Non-zero coefficients  ($|w_i| > 10^{-3}$)')
# ax2.set_title('Sparsity vs constraint budget')
# plt.suptitle('Effect of the $\\ell_1$ budget $t$', y=1.02)
# plt.tight_layout()
# ─────────────────────────────────────────────────────────────────────────────

---
## Discussion Questions

Answer the following questions in the cell below.

1. **Sparse iterates**: Why does Frank-Wolfe automatically produce sparse iterates, while PGD does not?
   *(Hint: think about the structure of the LMO solution for the ℓ₁ ball.)*

2. **Affine invariance**: The lecture notes state that Frank-Wolfe is affine invariant (Exercise 11, Problem 1).
   What does this mean, and why is it a useful property?
   Does PGD share this property?

3. **Convergence comparison**: Based on your convergence plots in Exercise 6,
   does PGD converge faster or slower than FW?
   Does the theoretical $O(1/k)$ bound for FW appear tight?

4. **Duality gap**: Does the duality gap track the true suboptimality closely?
   What would happen to the bound if the LMO were solved inexactly by a factor $\delta$?
   *(Hint: see Problem 4 in the exercise sheet — the rate becomes $2M(1+\delta)/(k+2)$.)*

5. **Sparsity–accuracy trade-off** (Bonus): Describe the pattern you observe in Exercise 8.
   How would you choose $t$ in a real application?